In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
import matplotlib.pyplot as plt
import timm
import cv2
from pathlib import Path
from tqdm.notebook import tqdm
import onnxruntime as ort
print(ort.get_available_providers())


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# Cell 1 — Configuration
# ============================================================
from dataclasses import dataclass 
from pathlib import Path

PROJECT_ROOT = Path("") #set your path

@dataclass
class DistillConfig:
    """All hyperparameters in one place."""
    # --- Paths ---
    images_dir: str = str(PROJECT_ROOT / "dataset_cliff/images") #path to cropped images from tools/annotate_dataset.py
    annotations_dir: str = str(PROJECT_ROOT / "dataset_cliff/annotations") #path to annotations from tools/annotate_dataset.py
    mhr_model_path: str = str(PROJECT_ROOT / "checkpoints/sam-3d-body-dinov3/assets/mhr_model.pt") #path to MHR model
    log_dir: str = str(PROJECT_ROOT / "runs/distill_repvit_cliff_v1") # Bumped to v1!

    # --- Data ---
    image_size: int = 224
    max_images: int = 100_000       
    val_split: float = 0.1
    num_workers: int = 4
    augment: bool = True

    # --- Model (Transformer + CLIFF) ---
    backbone: str = "repvit_m2_3"   
    backbone_feat_dim: int = 640    # RepViT m2_3 usually outputs 640
    d_model: int = 512              
    n_heads: int = 8                
    n_decoder_layers: int = 4       
    dropout: float = 0.1

    # --- Output dimensions ---
    pose_dim: int = 136             
    scale_dim: int = 68             
    shape_dim: int = 45             
    cam_dim: int = 3                
    cliff_dim: int = 3              
    num_joints: int = 70            # We track 70 joints

    @property
    def model_params_dim(self) -> int:
        return self.pose_dim + self.scale_dim  # 204

    # --- Training ---
    batch_size: int = 64
    lr: float = 3e-4
    weight_decay: float = 1e-4
    epochs: int = 300
    warmup_epochs: int = 3
    grad_clip: float = 5.0
    use_amp: bool = True            

    # --- Loss weights (UPDATED for HMR Balance) ---
    w_pose: float = 1.0
    w_scale: float = 0.1
    w_shape: float = 1.0
    w_cam: float = 0.1              
    w_keypoints3d: float = 10.0     # Bumped up: 3D joints are the strongest spatial anchor
    w_keypoints2d: float = 10.0     # Bumped up: Forces pixel-perfect alignment
    w_3d_joints: float = 1e-3
    w_structural: float = 0.01
    w_reproj_3d: float = 0.01  # High weight to enforce strict scale/camera alignment
    w_reproj_mhr: float = 0.01  # High weight to enforce strict scale/camera alignment

    # ==========================================================
    # Feet Barycenter Anchors
    # ==========================================================
    w_feet = 0.01  # Soft prior weight, matching w_structural
    
    # Native 70-joint format indices
    native_left_foot  = [15, 16, 17]
    native_right_foot = [18, 19, 20]
    
    # MHR 127-joint format indices
    mhr_left_foot     = [6, 7, 8] 
    mhr_right_foot    = [22, 23, 24]
    
    # Data-Driven Topology Mapping (Native 70 <-> MHR 127)
    native_mapping_ids = [ 1, 2, 5, 6, 9, 10, 11, 12, 13, 14, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62 ]
    mhr_mapping_ids    = [ 125, 123, 75, 39, 2, 18, 3, 19, 5, 21, 64, 63, 62, 61, 59, 58, 57, 56, 55, 54, 53, 52, 51, 50, 49, 48, 47, 46, 45, 44, 41, 100, 99, 98, 97, 95, 94, 93, 92, 91, 90, 89, 88, 87, 86, 85, 84, 83, 82, 81, 80, 77 ]

cfg = DistillConfig()
print(f"Config loaded. Backbone: {cfg.backbone}. Ready for V3 Training.")

In [ ]:
# ==============================================================================
# Cell 2 — X-Ray Dataset Visualizer (Math Verification)
# ==============================================================================
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- 1. Load Lightweight MHR Wrapper ---
class MHRForwardPass:
    def __init__(self, mhr_path, device):
        self.device = device
        self.mhr = torch.jit.load(mhr_path, map_location=device).eval()
        for p in self.mhr.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def get_joints(self, model_params, shape_params):
        B = model_params.shape[0]
        dummy_identity = torch.zeros(B, 45, device=self.device, dtype=model_params.dtype)
        concat_params = torch.cat([model_params, dummy_identity], dim=1)
        joint_params = self.mhr.character_torch.model_parameters_to_joint_parameters(concat_params)
        skel_state = self.mhr.character_torch.joint_parameters_to_skeleton_state(joint_params)
        return skel_state

mhr_module = MHRForwardPass(cfg.mhr_model_path, device)

# --- 2. Load a Random Sample ---
npz_files = list(Path(cfg.annotations_dir).glob("*.npz"))
sample_npz = random.choice(npz_files)
sample_stem = sample_npz.stem

img_path = Path(cfg.images_dir) / f"{sample_stem}.jpg"
if not img_path.exists():
    img_path = Path(cfg.images_dir) / f"{sample_stem}.png"

ann = np.load(str(sample_npz))
img_bgr = cv2.imread(str(img_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# --- 3. Extract Ground Truths ---
gt_2d_full = ann["joints_2d"]               
gt_3d_raw = ann["joints_3d"]                
model_params = ann["mhr_model_params"]      
shape_params = ann["shape_params"]          
cam_trans = ann["cam_trans"]                
focal_length = ann["cam_focal_length"]      
orig_shape = ann["orig_shape"]              
bbox_square = ann["bbox_square"]            

# --- 4. Forward Pass through MHR with 6D Math Injection ---
mp_tensor = torch.from_numpy(model_params).unsqueeze(0).to(device)
sp_tensor = torch.from_numpy(shape_params).unsqueeze(0).to(device)
raw_mhr_joints = mhr_module.get_joints(mp_tensor, sp_tensor)[0, :, :3].cpu().numpy()

# ------------------------------------------------------------------------------
# THE HOLY GRAIL MATH: Coordinate Transformations
# ------------------------------------------------------------------------------
# A. MHR -> Vision Coordinates
raw_mhr_joints = raw_mhr_joints / 100.0        
raw_mhr_joints[:, 1] = -raw_mhr_joints[:, 1]   
raw_mhr_joints[:, 2] = -raw_mhr_joints[:, 2]   

# B. SAM3D 70-Joints -> Camera Space
j3d_70_cam = gt_3d_raw + cam_trans 

# --- 5. Project 3D to Full-Frame 2D ---
orig_h, orig_w = orig_shape
principal_point = np.array([orig_w / 2.0, orig_h / 2.0])

mhr_cam = raw_mhr_joints + cam_trans
z_mhr = np.maximum(mhr_cam[:, 2], 0.01)
u_mhr_full = (focal_length[0] * mhr_cam[:, 0] / z_mhr) + principal_point[0]
v_mhr_full = (focal_length[1] * mhr_cam[:, 1] / z_mhr) + principal_point[1]

z_70 = np.maximum(j3d_70_cam[:, 2], 0.01)
u_70_full = (focal_length[0] * j3d_70_cam[:, 0] / z_70) + principal_point[0]
v_70_full = (focal_length[1] * j3d_70_cam[:, 1] / z_70) + principal_point[1]

# --- 6. Map Full-Frame 2D to 224x224 Crop Space ---
x1, y1, x2, y2 = bbox_square
crop_w, crop_h = x2 - x1, y2 - y1

u_mhr_crop = (u_mhr_full - x1) / max(crop_w, 1) * cfg.image_size
v_mhr_crop = (v_mhr_full - y1) / max(crop_h, 1) * cfg.image_size

u_70_crop = (u_70_full - x1) / max(crop_w, 1) * cfg.image_size
v_70_crop = (v_70_full - y1) / max(crop_h, 1) * cfg.image_size

gt_u_crop = (gt_2d_full[:, 0] - x1) / max(crop_w, 1) * cfg.image_size
gt_v_crop = (gt_2d_full[:, 1] - y1) / max(crop_h, 1) * cfg.image_size

# --- 7. Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(img_rgb)
axes[0].scatter(gt_u_crop, gt_v_crop, s=15, c='lime', edgecolors='black', label='Native 2D')
axes[0].scatter(u_70_crop, v_70_crop, s=40, facecolors='none', edgecolors='cyan', linewidths=1.5, label='Projected 3D (70pts)')
axes[0].set_title(f"Target 1: Native 2D vs Projected 3D Head\n{sample_stem}", fontsize=10)
axes[0].legend(loc='upper right')
axes[0].axis('off')

axes[1].imshow(img_rgb)
axes[1].scatter(u_mhr_crop, v_mhr_crop, s=15, c='red', edgecolors='black', marker='X')
axes[1].set_title(f"Target 2: Projected MHR Params\nMath verification (6D Cycle)", fontsize=10)
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 3 — Dataset and Data Augmentation
# ============================================================
import os
import glob
import random
import numpy as np
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.transforms.functional as F_t
from torchvision.transforms import InterpolationMode

class RandomPixelate:
    """Simulates low-quality webcams or low-bandwidth video."""
    def __init__(self, p=0.2, min_res=32, max_res=112):
        self.p = p
        self.min_res = min_res  
        self.max_res = max_res

    def __call__(self, img):
        if random.random() < self.p:
            orig_w, orig_h = img.size
            degraded_h = random.randint(self.min_res, self.max_res)
            degraded_w = int(orig_w * (degraded_h / orig_h))
            # Downsample then upsample to create pixelation effect
            img = F_t.resize(img, [degraded_h, degraded_w], interpolation=InterpolationMode.NEAREST)
            img = F_t.resize(img, [orig_h, orig_w], interpolation=InterpolationMode.NEAREST)
        return img
    

class SAM3DStudentDataset(Dataset):
    def __init__(self, images_dir: str, annotations_dir: str,
                 image_size: int = 224, max_images: int | None = None,
                 augment: bool = True):
        super().__init__()
        self.image_size = image_size
        self.images_dir = Path(images_dir)
        self.annotations_dir = Path(annotations_dir)

        # --- 1. Build Transform Pipeline ---
        # The images on disk are already 224x224, but Resize guarantees safety.
        tfm_list = [transforms.Resize((image_size, image_size))]
        
        if augment:
            tfm_list.extend([
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                RandomPixelate(p=0.2, min_res=48, max_res=112), 
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=5)], p=0.2), 
            ])
            
        tfm_list.extend([
            transforms.ToTensor(),
            # Standard ImageNet Normalization
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
        
        if augment:
            tfm_list.append(transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)))
            
        self.transform = transforms.Compose(tfm_list)

        # --- 2. Pair Discovery ---
        # Fast indexing of available NPZ files and corresponding images
        ann_files = list(self.annotations_dir.glob("*.npz"))
        
        self.pairs = []
        for npz_path in sorted(ann_files):
            stem = npz_path.stem
            # Check for jpg or png
            img_path = self.images_dir / f"{stem}.jpg"
            if not img_path.exists():
                img_path = self.images_dir / f"{stem}.png"
                
            if img_path.exists():
                self.pairs.append((img_path, npz_path))

        if max_images is not None:
            self.pairs = self.pairs[:max_images]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, npz_path = self.pairs[idx]

        # --- 3. Load and Augment Image ---
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image) # Returns float tensor [3, 224, 224]

        # --- 4. Load Annotations ---
        ann = np.load(npz_path)
        
        orig_h, orig_w = ann["orig_shape"]
        
        # --- 5. Correct Mathematical Rescaling for 2D Joints ---
        sq_bbox = ann["bbox_square"] 
        sq_x1, sq_y1 = sq_bbox[0], sq_bbox[1]
        orig_crop_size = sq_bbox[2] - sq_bbox[0]
        
        # Scale factor from the original crop square to the 224x224 tensor
        true_scale = self.image_size / max(orig_crop_size, 1.0)

        # Shift full-frame 2D joints relative to crop top-left, then scale
        joints_2d = ann["joints_2d"].copy()
        joints_2d[:, 0] = (joints_2d[:, 0] - sq_x1) * true_scale
        joints_2d[:, 1] = (joints_2d[:, 1] - sq_y1) * true_scale

        # NEW: Normalize to [-1, 1] for neural network stability!
        joints_2d[:, 0] = (joints_2d[:, 0] / self.image_size) * 2.0 - 1.0
        joints_2d[:, 1] = (joints_2d[:, 1] / self.image_size) * 2.0 - 1.0

        joints_2d = torch.from_numpy(joints_2d).float()

        # --- 6. CLIFF Conditioning Vectors ---
        # Using the tighter YOLO bbox to tell the network where the person actually is
        tight_bbox = ann["bbox"] 
        cx = (tight_bbox[0] + tight_bbox[2]) / 2.0
        cy = (tight_bbox[1] + tight_bbox[3]) / 2.0
        
        # Normalize to [-1, 1] space for better network stability
        cx_norm = 2.0 * (cx / orig_w) - 1.0
        cy_norm = 2.0 * (cy / orig_h) - 1.0
        
        # Scale relative to the longest edge of the original image
        b_size = max(tight_bbox[2] - tight_bbox[0], tight_bbox[3] - tight_bbox[1])
        b_scale = b_size / max(orig_w, orig_h)
        cliff_cond = torch.tensor([cx_norm, cy_norm, b_scale], dtype=torch.float32)

        return {
            "image": image,                                     # [3, 224, 224]
            "cliff_cond": cliff_cond,                           # [3]
            "mhr_model_params": torch.from_numpy(ann["mhr_model_params"]).float(), # [204]
            "shape_params": torch.from_numpy(ann["shape_params"]).float(),         # [45]
            "cam_trans": torch.from_numpy(ann["cam_trans"]).float(),               # [3]
            "cam_focal": torch.from_numpy(ann["cam_focal_length"]).float(),        # [2]
            "joints_2d": joints_2d,                             # [70, 2] Crop Space
            "joints_3d": torch.from_numpy(ann["joints_3d"]).float(),               # [70, 3] Root-centered
            "orig_shape": torch.tensor([orig_h, orig_w], dtype=torch.float32),      # [2]
            "bbox_square": sq_bbox
        }

# --- Initialization & Dataloaders ---
full_dataset = SAM3DStudentDataset(
    cfg.images_dir, 
    cfg.annotations_dir, 
    augment=cfg.augment, 
    max_images=cfg.max_images
)

val_dataset_clean = SAM3DStudentDataset(
    cfg.images_dir, 
    cfg.annotations_dir, 
    augment=False, 
    max_images=cfg.max_images
)

n_val = int(len(full_dataset) * cfg.val_split)
n_train = len(full_dataset) - n_val

# Use a generator for reproducible splits
generator = torch.Generator().manual_seed(42)
train_dataset, _ = random_split(full_dataset, [n_train, n_val], generator=generator)

generator = torch.Generator().manual_seed(42)
_, val_dataset = random_split(val_dataset_clean, [n_train, n_val], generator=generator)

train_loader = DataLoader(
    train_dataset, 
    batch_size=cfg.batch_size, 
    shuffle=True, 
    num_workers=cfg.num_workers, 
    pin_memory=True, 
    drop_last=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=cfg.batch_size, 
    shuffle=False, 
    num_workers=cfg.num_workers, 
    pin_memory=True
)


print(f"Dataset Loaded! Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# ==============================================================================
# Cell 4 — CLIFF & Augmentation Sanity Check
# ==============================================================================
import matplotlib.pyplot as plt
import cv2

# 1. Grab a single batch from the DataLoader
sample_batch = next(iter(train_loader))

# Extract the first item in the batch
idx = 0
img_tensor = sample_batch["image"][idx]
# Un-normalize the 2D joints for visualization ---
joints_2d_norm = sample_batch["joints_2d"][idx].numpy()  # Currently in [-1, 1] space
joints_2d_crop = ((joints_2d_norm + 1.0) / 2.0) * cfg.image_size  # Map back to [0, 224] pixel space

# Get MHR parameters and Camera data
mhr_params = sample_batch["mhr_model_params"][idx:idx+1].to(device)
shape_params = sample_batch["shape_params"][idx:idx+1].to(device)
cam_trans = sample_batch["cam_trans"][idx].numpy()
cam_focal = sample_batch["cam_focal"][idx].numpy()       # Original UNTOUCHED focal length

# Get CLIFF Conditioning & Geometry data
cliff_cond = sample_batch["cliff_cond"][idx].numpy()
orig_shape = sample_batch["orig_shape"][idx].numpy()     # [H, W]
bbox_square = sample_batch["bbox_square"][idx].numpy()   # [x1, y1, x2, y2]

orig_h, orig_w = orig_shape[0], orig_shape[1]

# ------------------------------------------------------------------------------
# SANITY CHECK 1: Reverse-Engineer CLIFF Conditioning
# ------------------------------------------------------------------------------
# cliff_cond = [cx_norm, cy_norm, scale]
cx_full = ((cliff_cond[0] + 1.0) / 2.0) * orig_w
cy_full = ((cliff_cond[1] + 1.0) / 2.0) * orig_h
box_size_full = cliff_cond[2] * max(orig_h, orig_w)

print("--- CLIFF Conditioning Verification ---")
print(f"Original Image Size: {orig_w}x{orig_h}")
print(f"BBox Center in Full Image: X={cx_full:.1f}, Y={cy_full:.1f}")
print(f"BBox Max Dimension: {box_size_full:.1f} pixels")
print(f"Does this look like a human bounding box? (Yes, if center is within image bounds)")
print("---------------------------------------")

# 2. Un-normalize the image tensor for viewing
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_unnorm = img_tensor * std + mean
img_np = img_unnorm.permute(1, 2, 0).numpy().clip(0, 1)

# 3. Generate 3D Joints using MHR
raw_3d_joints = mhr_module.get_joints(mhr_params, shape_params)[0, :, :3].cpu().numpy()
raw_3d_joints = raw_3d_joints / 100.0          # cm to meters
raw_3d_joints[:, 1] = -raw_3d_joints[:, 1]     # Y-down convention
raw_3d_joints[:, 2] = -raw_3d_joints[:, 2]     # Z-forward convention

# ------------------------------------------------------------------------------
# SANITY CHECK 2: The CLIFF Projection Math
# ------------------------------------------------------------------------------
# A. Project natively into the FULL FRAME space
j3d_cam = raw_3d_joints + cam_trans
z = np.maximum(j3d_cam[:, 2], 0.01)

principal_point_x = orig_w / 2.0
principal_point_y = orig_h / 2.0

u_full = (cam_focal[0] * j3d_cam[:, 0] / z) + principal_point_x
v_full = (cam_focal[1] * j3d_cam[:, 1] / z) + principal_point_y

# B. Map the full-frame 2D points into the 224x224 crop space
sq_x1, sq_y1, sq_x2, sq_y2 = bbox_square
crop_w = sq_x2 - sq_x1
crop_h = sq_y2 - sq_y1
true_scale = 224.0 / max(crop_w, 1.0)

u_crop_proj = (u_full - sq_x1) * true_scale
v_crop_proj = (v_full - sq_y1) * true_scale

# 4. Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Plot 1: The Native 2D labels from the dataset (Checks augmentation/mapping)
axes[0].imshow(img_np)
axes[0].scatter(joints_2d_crop[:, 0], joints_2d_crop[:, 1], s=15, c='lime', edgecolors='black', label='Native 2D')
axes[0].set_title(f"Target 1: Native 2D Joints\n(Verifies Augmentations & Crop mapping)", fontsize=10)
axes[0].legend(loc='upper right')
axes[0].axis('off')

# Plot 2: The 3D -> 2D CLIFF Projection (Checks focal length & cam_trans)
axes[1].imshow(img_np)
axes[1].scatter(u_crop_proj, v_crop_proj, s=15, c='red', edgecolors='black', marker='X')
axes[1].set_title(f"Target 2: CLIFF Projected 3D MHR\n(Verifies Full-Frame Camera Math)", fontsize=10)
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Cell 5 — Data-Driven Topology Discovery (Pixel-Space Edition - FIXED)
# ==============================================================================
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from tqdm import tqdm

print("--- Starting 2D Pixel-Space Topology Discovery ---")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mhr_module = MHRForwardPass(cfg.mhr_model_path, device)

# --- Configuration ---
NUM_BATCHES_TO_SCAN = 100  
PIXEL_THRESHOLD = 1.0  # 15 pixels is a great sweet spot for 224x224 anatomical matches

cumulative_dist_matrix = torch.zeros((cfg.num_joints, 127), device=device)
total_samples = 0

print(f"Scanning {NUM_BATCHES_TO_SCAN} batches to compute average 2D pixel distances...")
train_iter = iter(train_loader)

for _ in tqdm(range(NUM_BATCHES_TO_SCAN)):
    batch = next(train_iter)
    batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
    B = batch["mhr_model_params"].shape[0]
    
    # 1. Un-normalize Native 2D to Crop Space [0, 224]
    native_2d_norm = batch["joints_2d"][..., :2] 
    native_2d_crop = ((native_2d_norm + 1.0) / 2.0) * cfg.image_size 
    
    # 2. Extract 3D MHR Joints
    with torch.no_grad():
        mhr_3d_raw = mhr_module.get_joints(batch["mhr_model_params"], batch["shape_params"])[..., :3]
    
    mhr_3d = mhr_3d_raw / 100.0          
    mhr_3d[..., 1] = -mhr_3d[..., 1]     
    mhr_3d[..., 2] = -mhr_3d[..., 2]     
    
    # 3. Apply CLIFF Projection Math (Batched properly!)
    cam_trans = batch["cam_trans"].unsqueeze(1) # [B, 1, 3]
    j3d_cam = mhr_3d + cam_trans # [B, 127, 3]
    z = torch.clamp(j3d_cam[..., 2], min=0.01) # [B, 127]
    
    # FIXED: Removed the buggy .unsqueeze(1) so shapes match [B, 1] * [B, 127]
    orig_shape = batch["orig_shape"]
    orig_w, orig_h = orig_shape[:, 1:2], orig_shape[:, 0:1] # [B, 1]
    cx, cy = orig_w / 2.0, orig_h / 2.0 # [B, 1]
    
    focal_key = "cam_focal" if "cam_focal" in batch else "cam_focal_length"
    cam_focal = batch[focal_key]
    fx, fy = cam_focal[:, 0:1], cam_focal[:, 1:2] # [B, 1]
    
    u_full = (fx * j3d_cam[..., 0] / z) + cx # [B, 127]
    v_full = (fy * j3d_cam[..., 1] / z) + cy # [B, 127]
    
    bbox = batch["bbox_square"]
    sq_x1, sq_y1, sq_x2 = bbox[:, 0:1], bbox[:, 1:2], bbox[:, 2:3] # [B, 1]
    crop_w = torch.clamp(sq_x2 - sq_x1, min=1.0) # [B, 1]
    true_scale = cfg.image_size / crop_w # [B, 1]
    
    u_crop = (u_full - sq_x1) * true_scale # [B, 127]
    v_crop = (v_full - sq_y1) * true_scale # [B, 127]
    mhr_2d_crop = torch.stack([u_crop, v_crop], dim=-1) # [B, 127, 2]
    
    # 4. Compute Pairwise Distance in Pixel Space!
    dist_matrix = torch.cdist(native_2d_crop, mhr_2d_crop) # [B, 70, 127]
    cumulative_dist_matrix += dist_matrix.sum(dim=0)
    total_samples += B

# Average the distance matrix
avg_dist_matrix = cumulative_dist_matrix / total_samples

# --- Extract the Mapping ---
min_dists, closest_mhr_indices = avg_dist_matrix.min(dim=1)

valid_mask = min_dists < PIXEL_THRESHOLD
native_mapping_ids = torch.arange(70, device=device)[valid_mask].cpu().numpy()
mhr_mapping_ids = closest_mhr_indices[valid_mask].cpu().numpy()
matched_distances = min_dists[valid_mask].cpu().numpy()

native_ids_2d, mhr_ids_2d = native_mapping_ids, mhr_mapping_ids 
print("\n--- Discovery Complete ---")
print(f"Found {len(native_mapping_ids)} valid anchor pairs with average distance < {PIXEL_THRESHOLD} pixels.")

# ==============================================================================
# VISUALIZATION: Plotting the Anchors on a Sample
# ==============================================================================
sample_idx = 0
nat_2d_viz = native_2d_crop[sample_idx].cpu().numpy()
mhr_2d_viz = mhr_2d_crop[sample_idx].cpu().numpy()

# Un-normalize image
img_tensor = batch["image"][sample_idx].cpu()
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_unnorm = img_tensor * std + mean
img_rgb = img_unnorm.permute(1, 2, 0).numpy().clip(0, 1)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img_rgb)
ax.set_title(f"Data-Driven Anchor Mapping (Thresh: {PIXEL_THRESHOLD} px)", fontsize=14)

# Plot all joints faintly
ax.scatter(nat_2d_viz[:, 0], nat_2d_viz[:, 1], s=20, c='cyan', alpha=0.3, label='All Native')
ax.scatter(mhr_2d_viz[:, 0], mhr_2d_viz[:, 1], s=20, c='red', alpha=0.3, label='All MHR')

# Draw connections for the valid mapped anchors
for n_idx, m_idx in zip(native_mapping_ids, mhr_mapping_ids):
    ax.scatter(nat_2d_viz[n_idx, 0], nat_2d_viz[n_idx, 1], s=50, c='lime', edgecolors='black', zorder=5)
    ax.scatter(mhr_2d_viz[m_idx, 0], mhr_2d_viz[m_idx, 1], s=50, c='yellow', marker='X', edgecolors='black', zorder=5)
    
    line = mlines.Line2D([nat_2d_viz[n_idx, 0], mhr_2d_viz[m_idx, 0]], 
                         [nat_2d_viz[n_idx, 1], mhr_2d_viz[m_idx, 1]], 
                         color='magenta', linewidth=1.5, zorder=4)
    ax.add_line(line)

# Clean up legend
handles, labels = ax.get_legend_handles_labels()
match_marker = mlines.Line2D([], [], color='magenta', marker='o', markersize=8, markerfacecolor='lime', label='Discovered Anchors')
ax.legend(handles=[handles[0], handles[1], match_marker], loc='upper right')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Cell 6 — Data-Driven Topology Discovery (3D Meter Space - FIXED ALIGNMENT)
# ==============================================================================
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.lines as mlines
from tqdm import tqdm

print("--- Starting 3D Topology Discovery ---")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mhr_module = MHRForwardPass(cfg.mhr_model_path, device)

# --- Configuration ---
NUM_BATCHES_TO_SCAN = 100  
# 4 cm is a rigorous but realistic threshold for overlapping anatomical joints
DISTANCE_THRESHOLD_METERS = 0.01 

cumulative_dist_matrix = torch.zeros((cfg.num_joints, 127), device=device)
total_samples = 0

print(f"Scanning {NUM_BATCHES_TO_SCAN} batches to compute true 3D distances...")
train_iter = iter(train_loader)

for _ in tqdm(range(NUM_BATCHES_TO_SCAN)):
    batch = next(train_iter)
    batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
    B = batch["mhr_model_params"].shape[0]
    
    # 1. Extract 3D Native Joints (Body-Centered, Meters, Y-Down, Z-Forward)
    native_3d = batch["joints_3d"][..., :3] 
    
    # 2. Extract 3D MHR Joints (Internal coords, cm)
    with torch.no_grad():
        mhr_3d_raw = mhr_module.get_joints(batch["mhr_model_params"], batch["shape_params"])[..., :3]
    
    # Convert MHR to the exact same frame as Native 3D!
    mhr_3d = mhr_3d_raw / 100.0          
    mhr_3d[..., 1] = -mhr_3d[..., 1]     
    mhr_3d[..., 2] = -mhr_3d[..., 2]     
    
    # NO MEAN CENTERING! They are already in the same coordinate space!
    
    # 3. Compute Pairwise Distance in 3D METER Space [B, 70, 127]
    dist_matrix = torch.cdist(native_3d, mhr_3d) 
    cumulative_dist_matrix += dist_matrix.sum(dim=0)
    total_samples += B

# Average the distance matrix
avg_dist_matrix = cumulative_dist_matrix / total_samples

# --- Extract the Mapping ---
min_dists, closest_mhr_indices = avg_dist_matrix.min(dim=1)

valid_mask = min_dists < DISTANCE_THRESHOLD_METERS
native_mapping_ids = torch.arange(70, device=device)[valid_mask].cpu().numpy()
mhr_mapping_ids = closest_mhr_indices[valid_mask].cpu().numpy()
matched_distances = min_dists[valid_mask].cpu().numpy()

native_ids_3d, mhr_ids_3d = native_mapping_ids, mhr_mapping_ids 
print("\n--- Discovery Complete ---")
print(f"Found {len(native_mapping_ids)} valid anchor pairs with average distance < {DISTANCE_THRESHOLD_METERS} meters.")

# ==============================================================================
# VISUALIZATION: Interactive 3D Plot with Plotly
# ==============================================================================
import plotly.graph_objects as go
import numpy as np

sample_idx = 0
# Extract raw coordinates
nat_3d_viz = native_3d[sample_idx].cpu().numpy()
mhr_3d_viz = mhr_3d[sample_idx].cpu().numpy()

# To make the person stand upright in Plotly (where Z is usually up):
# Data X -> Plotly X
# Data Z (Forward) -> Plotly Y (Depth)
# Data -Y (Down) -> Plotly Z (Height)
def map_coords(pts):
    return pts[:, 0], pts[:, 2], -pts[:, 1]

nx, ny, nz = map_coords(nat_3d_viz)
mx, my, mz = map_coords(mhr_3d_viz)

fig = go.Figure()

# 1. Plot Unmatched Native Joints (Cyan)
fig.add_trace(go.Scatter3d(
    x=nx, y=ny, z=nz,
    mode='markers',
    marker=dict(size=4, color='cyan', opacity=0.4),
    name='All Native (70)',
    text=[f"Native Joint: {i}" for i in range(70)],
    hoverinfo='text'
))

# 2. Plot Unmatched MHR Joints (Red)
fig.add_trace(go.Scatter3d(
    x=mx, y=my, z=mz,
    mode='markers',
    marker=dict(size=4, color='red', opacity=0.4),
    name='All MHR (127)',
    text=[f"MHR Joint: {i}" for i in range(127)],
    hoverinfo='text'
))

# 3. Plot Matched Anchors
# Extract coordinates for the matched pairs
nat_anchors_x, nat_anchors_y, nat_anchors_z = nx[native_mapping_ids], ny[native_mapping_ids], nz[native_mapping_ids]
mhr_anchors_x, mhr_anchors_y, mhr_anchors_z = mx[mhr_mapping_ids], my[mhr_mapping_ids], mz[mhr_mapping_ids]

# Native Anchors (Green)
fig.add_trace(go.Scatter3d(
    x=nat_anchors_x, y=nat_anchors_y, z=nat_anchors_z,
    mode='markers',
    marker=dict(size=7, color='lime', line=dict(color='black', width=1)),
    name='Native Anchors',
    text=[f"Matched Native: {i}" for i in native_mapping_ids],
    hoverinfo='text'
))

# MHR Anchors (Yellow)
fig.add_trace(go.Scatter3d(
    x=mhr_anchors_x, y=mhr_anchors_y, z=mhr_anchors_z,
    mode='markers',
    marker=dict(size=7, color='yellow', symbol='cross', line=dict(color='black', width=1)),
    name='MHR Anchors',
    text=[f"Matched MHR: {i}" for i in mhr_mapping_ids],
    hoverinfo='text'
))

# 4. Draw Connection Lines (Magenta)
# To draw multiple disconnected lines efficiently in Plotly, we interleave with None
line_x, line_y, line_z = [], [], []
for i in range(len(native_mapping_ids)):
    line_x.extend([nat_anchors_x[i], mhr_anchors_x[i], None])
    line_y.extend([nat_anchors_y[i], mhr_anchors_y[i], None])
    line_z.extend([nat_anchors_z[i], mhr_anchors_z[i], None])

fig.add_trace(go.Scatter3d(
    x=line_x, y=line_y, z=line_z,
    mode='lines',
    line=dict(color='magenta', width=4),
    name='Physical Distance',
    hoverinfo='none'
))

# 5. Layout and Camera Setup
fig.update_layout(
    title=f"Interactive 3D Anchor Mapping (Thresh: {DISTANCE_THRESHOLD_METERS*100:.1f} cm)",
    scene=dict(
        xaxis_title='X (Meters)',
        yaxis_title='Z Depth (Meters)',
        zaxis_title='Y Height (Meters)',
        aspectmode='data' # THIS IS CRITICAL: Keeps the human proportions physically accurate!
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(x=0.8, y=0.9)
)

fig.show()

print("--- Anchor Verification: 2D vs 3D ---")

# 1. Create sets of tuples for easy comparison
mapping_2d = set(zip(native_ids_2d, mhr_ids_2d))
mapping_3d = set(zip(native_ids_3d, mhr_ids_3d))

# 2. Check for perfect agreement
if mapping_2d == mapping_3d:
    print("✅ PERFECT MATCH! Both 2D and 3D discovery found the exact same 52 anchor pairs.")
else:
    print("⚠️ MISMATCH DETECTED. The lists differ slightly.")
    
    # Find what's unique to 2D
    only_in_2d = mapping_2d - mapping_3d
    if only_in_2d:
        print(f"\nPairs found ONLY in 2D ({len(only_in_2d)}):")
        for nat, mhr in sorted(only_in_2d):
            print(f"  Native [{nat:02d}] <--> MHR [{mhr:03d}]")
            
    # Find what's unique to 3D
    only_in_3d = mapping_3d - mapping_2d
    if only_in_3d:
        print(f"\nPairs found ONLY in 3D ({len(only_in_3d)}):")
        for nat, mhr in sorted(only_in_3d):
            print(f"  Native [{nat:02d}] <--> MHR [{mhr:03d}]")

# 3. Print the final agreed-upon list to copy into your config!
agreed_mapping = sorted(list(mapping_2d.intersection(mapping_3d)))

print("\n--- Final Verified Anchor Dictionary ---")
print("Copy this into your DistillConfig:")
print("native_mapping_ids = [", ", ".join(str(pair[0]) for pair in agreed_mapping), "]")
print("mhr_mapping_ids    = [", ", ".join(str(pair[1]) for pair in agreed_mapping), "]")

In [ ]:
# ============================================================
# Cell 7 — Student Architecture (Mini-SAM3D with PE)
# ============================================================
import torch
import torch.nn as nn
import timm

def get_2d_sincos_pos_embed(embed_dim, grid_size):
    """Generate 2D sine-cosine positional embedding for preserving spatial topology."""
    grid_h, grid_w = grid_size, grid_size
    grid_y, grid_x = torch.meshgrid(torch.arange(grid_h), torch.arange(grid_w), indexing='ij')
    
    omega = torch.arange(embed_dim // 4).float() / (embed_dim // 4)
    omega = 1.0 / (10000 ** omega)
    
    out_y = grid_y.flatten().unsqueeze(1) * omega.unsqueeze(0)
    out_x = grid_x.flatten().unsqueeze(1) * omega.unsqueeze(0)
    
    pe_y = torch.cat([torch.sin(out_y), torch.cos(out_y)], dim=1)
    pe_x = torch.cat([torch.sin(out_x), torch.cos(out_x)], dim=1)
    
    pos_embed = torch.cat([pe_y, pe_x], dim=1) # (H*W, embed_dim)
    return pos_embed.unsqueeze(0) # (1, H*W, embed_dim)


class InstantHMRStudent(nn.Module):
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        self.cfg = cfg
        
        # 1. The Vision Backbone (RepViT)
        self.backbone = timm.create_model(cfg.backbone, pretrained=pretrained, num_classes=0)
        embed_dim = self.backbone.num_features 
        self.feat_proj = nn.Linear(embed_dim, cfg.d_model)
        
        # --- NEW: Positional Embedding for Transformer Memory ---
        # RepViT naturally downsamples by 32x. For a 224x224 image, the grid is 7x7.
        self.grid_size = cfg.image_size // 32
        pos_embed = get_2d_sincos_pos_embed(cfg.d_model, self.grid_size)
        # register_buffer ensures it moves to GPU automatically but isn't updated by the optimizer
        self.register_buffer("mem_pos_embed", pos_embed) 
        
        # 2. Learnable Query Tokens (1 Global + 70 2D + 70 3D = 141 Queries)
        self.num_global = 1
        self.num_2d = cfg.num_joints
        self.num_3d = cfg.num_joints
        self.total_queries = self.num_global + self.num_2d + self.num_3d
        
        # self.query_embed = nn.Parameter(torch.randn(1, self.total_queries, cfg.d_model))
        self.query_embed = nn.Parameter(torch.randn(1, self.total_queries, cfg.d_model) * 0.02)
        
        # CLIFF Conditioning Injector
        self.cond_proj = nn.Sequential(
            nn.Linear(cfg.cliff_dim, 128),
            nn.GELU(),
            nn.Linear(128, cfg.d_model)
        )
        
        # 3. Transformer Cross-Attention Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.d_model * 4,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=cfg.n_decoder_layers)
        
        # 4. The Output Heads
        
        # Head A: Global Head (Token 0 -> MHR Params + Shape Params + Camera)
        self.global_out_dim = cfg.model_params_dim + cfg.shape_dim + cfg.cam_dim
        self.head_global = nn.Linear(cfg.d_model, self.global_out_dim)
        
        # --- Safely Split 2D Head ---
        self.head_2d_feat = nn.Sequential(
            nn.Linear(cfg.d_model, 256),
            nn.GELU()
        )
        self.head_2d_xy = nn.Sequential(
            nn.Linear(256, 2)
            # nn.Tanh() # Bounds predictions to [-1, 1]!
        )
        if getattr(cfg, 'learn_uncertainty', False):
            # Uncertainty must remain unbounded!
            self.head_2d_unc = nn.Linear(256, 1) 
        
        # Head C: 3D Keypoint Head (No Tanh here, 3D meters can exceed [-1, 1])
        self.dim_3d = 4 if getattr(cfg, 'learn_uncertainty', False) else 3
        self.head_3d = nn.Sequential(
            nn.Linear(cfg.d_model, 256),
            nn.GELU(),
            nn.Linear(256, self.dim_3d)
        )
        
        # Initialize camera Z translation
        nn.init.constant_(self.head_global.bias[-1], 2.0)
        nn.init.constant_(self.head_global.bias[-2], 0.0)
        nn.init.constant_(self.head_global.bias[-3], 0.0)

        # ==========================================================
        # HMR-Standard Zero-Initialization for Stability
        # ==========================================================
        # 1. Global Head (MHR Params) - Must start near neutral pose!
        nn.init.normal_(self.head_global.weight, mean=0.0, std=1e-4)
        # Zero out the biases (except the last 3 camera params we just set)
        nn.init.constant_(self.head_global.bias[:-3], 0.0)
        
        # 2. 2D Spatial Head - Start with tiny weights
        nn.init.normal_(self.head_2d_xy[0].weight, mean=0.0, std=1e-4)
        nn.init.constant_(self.head_2d_xy[0].bias, 0.0)
        
        # 3. 3D Spatial Head - Start with tiny weights
        nn.init.normal_(self.head_3d[-1].weight, mean=0.0, std=1e-4)
        nn.init.constant_(self.head_3d[-1].bias, 0.0)

    def forward(self, images, cliff_cond):
        B = images.shape[0]
        
        # 1. Extract spatial image features
        img_feats = self.backbone.forward_features(images)
        
        # Defensive flattening 
        if img_feats.dim() == 4:
            img_feats = img_feats.flatten(2).transpose(1, 2) # [B, H*W, C]
            
        memory = self.feat_proj(img_feats) 
        
        # --- NEW: Inject Spatial Topology into Memory ---
        # Broadcasting [1, H*W, d_model] across Batch size
        memory = memory + self.mem_pos_embed 
        
        # 2. Prepare Query Tokens [B, 141, d_model]
        queries = self.query_embed.expand(B, -1, -1)
        
        cond = self.cond_proj(cliff_cond).unsqueeze(1)
        queries = queries + cond
        
        # 3. Cross Attention
        shared_feats = self.transformer(tgt=queries, memory=memory)
        
        # 4. Route tokens to their dedicated heads
        feat_global = shared_feats[:, 0, :]                     
        feat_2d     = shared_feats[:, 1 : 1 + self.num_2d, :]   
        feat_3d     = shared_feats[:, 1 + self.num_2d :, :]     
        
        # --- Decode Global Token ---
        global_preds = self.head_global(feat_global)            
        
        idx_mhr = self.cfg.model_params_dim
        idx_shape = idx_mhr + self.cfg.shape_dim
        
        pred_mhr_params = global_preds[:, :idx_mhr]
        pred_shape_params = global_preds[:, idx_mhr : idx_shape]
        pred_cam_trans = global_preds[:, idx_shape :]
        
        # --- Decode 2D Spatial Tokens (Safely!) ---
        feat_2d_processed = self.head_2d_feat(feat_2d)
        pred_xy = self.head_2d_xy(feat_2d_processed) # [B, 70, 2] bounded by Tanh
        
        if getattr(self.cfg, 'learn_uncertainty', False):
            pred_unc = self.head_2d_unc(feat_2d_processed) # [B, 70, 1] unbounded
            pred_joints_2d = torch.cat([pred_xy, pred_unc], dim=-1) # [B, 70, 3]
        else:
            pred_joints_2d = pred_xy
            
        # --- Decode 3D Spatial Tokens ---
        pred_joints_3d = self.head_3d(feat_3d)                  
        
        return {
            "mhr_params": pred_mhr_params,
            "shape_params": pred_shape_params,
            "cam_trans": pred_cam_trans,
            "joints_2d": pred_joints_2d,
            "joints_3d": pred_joints_3d
        }

# --- Quick Architecture Sanity Check ---
if __name__ == "__main__":
    test_model = InstantHMRStudent(cfg, pretrained=False)
    dummy_img = torch.randn(2, 3, 224, 224)
    dummy_cliff = torch.randn(2, 3)
    
    out = test_model(dummy_img, dummy_cliff)
    print("--- Architecture Output Shapes ---")
    for k, v in out.items():
        print(f"{k}: {v.shape}")

In [ ]:
# ============================================================
# Cell 8 — The Distillation Loss Module (Foolproof Edition)
# ============================================================
import torch
import torch.nn as nn

# --- Modified Forward Pass (Differentiable) ---
class MHRForwardPass:
    def __init__(self, mhr_path, device):
        self.device = device
        self.mhr = torch.jit.load(mhr_path, map_location=device).eval()
        for p in self.mhr.parameters():
            p.requires_grad_(False)

    def get_joints(self, model_params, shape_params):
        B = model_params.shape[0]
        dummy_identity = torch.zeros(B, 45, device=model_params.device, dtype=model_params.dtype)
        concat_params = torch.cat([model_params, dummy_identity], dim=1)
        
        joint_params = self.mhr.character_torch.model_parameters_to_joint_parameters(concat_params.to(self.device))
        skel_state = self.mhr.character_torch.joint_parameters_to_skeleton_state(joint_params)
        
        return skel_state.to(model_params.device)

class DistillationLoss(nn.Module):
    def __init__(self, cfg, mhr_module):
        super().__init__()
        self.cfg = cfg
        self.mhr_module = mhr_module 
        self.mse = nn.MSELoss()
        self.l1 = nn.SmoothL1Loss()
        self.strict_l1 = nn.L1Loss()

    def _project_3d_to_norm_2d(self, j3d, pred_cam_trans, targets):
            """Helper to project 3D points (in meters) to normalized [-1, 1] 2D crop space."""
            # A. Move to Camera Space
            j3d_cam = j3d + pred_cam_trans.unsqueeze(1)
            
            # Prevent 1/Z^2 Gradient Explosions
            # 0.01 allows a gradient multiplier of 10,000. 
            # 0.4 meters (40cm) limits the multiplier to a safe limit, protecting the kinematic tree.
            z_cam = torch.clamp(j3d_cam[..., 2], min=0.4)
            
            # B. Extract Intrinsics
            fx = targets["cam_focal"][:, 0:1]       # Shape: [B, 1]
            fy = targets["cam_focal"][:, 1:2]       # Shape: [B, 1]
            cx = targets["orig_shape"][:, 1:2] / 2.0 # Shape: [B, 1]
            cy = targets["orig_shape"][:, 0:1] / 2.0 # Shape: [B, 1]
            
            # C. Full-Frame Projection
            u_full = (fx * j3d_cam[..., 0] / z_cam) + cx
            v_full = (fy * j3d_cam[..., 1] / z_cam) + cy
            
            # D. Map to Crop Space
            bbox = targets["bbox_square"]
            sq_x1 = bbox[:, 0:1] # [B, 1]
            sq_y1 = bbox[:, 1:2] # [B, 1]
            crop_w = torch.clamp(bbox[:, 2:3] - bbox[:, 0:1], min=1.0) # [B, 1]
            
            true_scale = self.cfg.image_size / crop_w
            u_crop = (u_full - sq_x1) * true_scale
            v_crop = (v_full - sq_y1) * true_scale
            
            # E. Normalize to [-1, 1]
            u_norm = (u_crop / self.cfg.image_size) * 2.0 - 1.0
            v_norm = (v_crop / self.cfg.image_size) * 2.0 - 1.0
            
            return torch.stack([u_norm, v_norm], dim=-1) # Shape: [B, J, 2]

    def forward(self, preds, targets):
        losses = {}
        idx_pose = self.cfg.pose_dim
        
        pred_pose = preds["mhr_params"][:, :idx_pose]
        pred_scale = preds["mhr_params"][:, idx_pose:]
        tgt_pose = targets["mhr_model_params"][:, :idx_pose]
        tgt_scale = targets["mhr_model_params"][:, idx_pose:]
        
        # --- 1. Base Parameters ---
        losses['loss_pose'] = self.l1(pred_pose, tgt_pose) * self.cfg.w_pose
        losses['loss_scale'] = self.mse(pred_scale, tgt_scale) * self.cfg.w_scale
        losses['loss_shape'] = self.mse(preds["shape_params"], targets["shape_params"]) * self.cfg.w_shape
        losses['loss_cam'] = self.mse(preds["cam_trans"], targets["cam_trans"]) * self.cfg.w_cam
        
        # --- 2. Differentiable 3D Joint Alignment (Centimeter space corrected!) ---
        pred_mhr_joints = self.mhr_module.get_joints(preds["mhr_params"], preds["shape_params"])[..., :3]
        with torch.no_grad():
            tgt_mhr_joints = self.mhr_module.get_joints(targets["mhr_model_params"], targets["shape_params"])[..., :3]
            
        losses['loss_mhr_joints'] = self.l1(pred_mhr_joints, tgt_mhr_joints.detach()) * self.cfg.w_3d_joints

        # --- Base MHR Vision Coordinates (Needed for Structural & Reprojection) ---
        pred_mhr_vision = pred_mhr_joints.clone() / 100.0
        pred_mhr_vision[..., 1] = -pred_mhr_vision[..., 1]
        pred_mhr_vision[..., 2] = -pred_mhr_vision[..., 2]

        # ==============================================================================
        # 2.5 Structural Consistency Loss (MHR mesh vs Native Keypoints)
        # ==============================================================================
        if hasattr(self.cfg, 'native_mapping_ids') and hasattr(self.cfg, 'mhr_mapping_ids'):
            pred_native_anchors = preds["joints_3d"][:, self.cfg.native_mapping_ids, :3]
            pred_mhr_anchors = pred_mhr_vision[:, self.cfg.mhr_mapping_ids, :3]
            losses['loss_structural'] = self.l1(pred_native_anchors, pred_mhr_anchors) * self.cfg.w_structural

        # ==============================================================================
        # Feet Barycenter Alignment Loss
        # ==============================================================================
        if hasattr(self.cfg, 'native_left_foot') and hasattr(self.cfg, 'w_feet'):
            nat_lf_pts = preds["joints_3d"][:, self.cfg.native_left_foot, :3]
            nat_rf_pts = preds["joints_3d"][:, self.cfg.native_right_foot, :3]
            
            mhr_lf_pts = pred_mhr_vision[:, self.cfg.mhr_left_foot, :3]
            mhr_rf_pts = pred_mhr_vision[:, self.cfg.mhr_right_foot, :3]
            
            nat_lf_bary = nat_lf_pts.mean(dim=1)
            nat_rf_bary = nat_rf_pts.mean(dim=1)
            mhr_lf_bary = mhr_lf_pts.mean(dim=1)
            mhr_rf_bary = mhr_rf_pts.mean(dim=1)
            
            losses['loss_feet'] = (self.strict_l1(nat_lf_bary, mhr_lf_bary) + self.strict_l1(nat_rf_bary, mhr_rf_bary)) * self.cfg.w_feet

        # --- 3. DYNAMIC Native Keypoints ---
        pred_2d = preds["joints_2d"]
        tgt_2d = targets["joints_2d"]
        losses['loss_2d_native'] = self.l1(pred_2d[..., :2], tgt_2d) * self.cfg.w_keypoints2d
            
        pred_3d = preds["joints_3d"]
        tgt_3d = targets["joints_3d"]
        losses['loss_3d_native'] = self.l1(pred_3d[..., :3], tgt_3d) * self.cfg.w_keypoints3d

        # ==============================================================================
        # 4. 2D Reprojection Loss (Camera-Pose Consistency)
        # ==============================================================================
        # A. Native 3D Head Reprojection (All 70 Joints)
        pred_reproj_native = self._project_3d_to_norm_2d(preds["joints_3d"][..., :3], preds["cam_trans"], targets)
        losses['loss_reproj_native'] = self.strict_l1(pred_reproj_native, tgt_2d) * self.cfg.w_reproj_3d

        # B. MHR Mesh Anchors Reprojection (52 Anchors)
        if hasattr(self.cfg, 'native_mapping_ids') and hasattr(self.cfg, 'mhr_mapping_ids'):
            # Only project the 52 matched anchors from the MHR mesh
            pred_mhr_anchors_vision = pred_mhr_vision[:, self.cfg.mhr_mapping_ids, :3]
            pred_reproj_mhr = self._project_3d_to_norm_2d(pred_mhr_anchors_vision, preds["cam_trans"], targets) #detach the MHR anchors to prevent gradients from flowing back through the projection during this loss calculation
            # pred_reproj_mhr = self._project_3d_to_norm_2d(pred_mhr_anchors_vision, preds["cam_trans"], targets)
            
            # Compare against the corresponding 52 ground truth 2D targets
            tgt_2d_anchors = tgt_2d[:, self.cfg.native_mapping_ids, :]
            losses['loss_reproj_mhr'] = self.strict_l1(pred_reproj_mhr, tgt_2d_anchors) * self.cfg.w_reproj_mhr
            
        losses['total_loss'] = sum(losses.values())
        return losses

# ==============================================================================
# SANITY CHECK: The Perfect Student
# ==============================================================================
if __name__ == "__main__":
    print("--- Running Loss Function Sanity Check ---")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mhr_module = MHRForwardPass(cfg.mhr_model_path, device)
    criterion = DistillationLoss(cfg, mhr_module)
    
    sample_batch_cpu = next(iter(train_loader))
    sample_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in sample_batch_cpu.items()}
    
    mock_preds = {
        "mhr_params": sample_batch["mhr_model_params"].clone(),
        "shape_params": sample_batch["shape_params"].clone(),
        "cam_trans": sample_batch["cam_trans"].clone(),
        "joints_2d": sample_batch["joints_2d"].clone(),
        "joints_3d": sample_batch["joints_3d"].clone()
    }
        
    losses = criterion(mock_preds, sample_batch)
    
    print("Losses for the Perfect Student:")
    for k, v in losses.items():
        print(f"  {k:<20}: {v.item():.6f}")
        
    if abs(losses['total_loss'].item()) < 1e-4:
        print("\n✅ SUCCESS: The Perfect Student achieved a loss of 0! Math is fully verified.")
    else:
        print("\n❌ ERROR: Loss is not zero. Check the math.")

In [ ]:
# ============================================================
# Cell 9 — The 1-Batch Overfit Test (VRAM Optimized)
# ============================================================
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import gc

print("--- Starting 1-Batch Overfit Test ---")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Flush Notebook Ghost Memory
gc.collect()
torch.cuda.empty_cache()

# 2. Instantiate Model, Loss, and Optimizer
overfit_model = InstantHMRStudent(cfg, pretrained=True).to(device)
criterion = DistillationLoss(cfg, mhr_module)

optimizer = optim.AdamW(overfit_model.parameters(), lr=1e-3, weight_decay=0.0)

# Setup Automatic Mixed Precision (AMP) Scaler to save VRAM
scaler = torch.amp.GradScaler(device='cuda', enabled=cfg.use_amp)

# 3. Grab exactly ONE batch and SLICE it to fit in VRAM
overfit_batch_cpu = next(iter(train_loader))

# We only need 8 images to prove the network can learn!
def slice_and_move(batch_dict, dev, subset_size=8):
    return {k: v[:subset_size].to(dev) if isinstance(v, torch.Tensor) else v for k, v in batch_dict.items()}

overfit_batch = slice_and_move(overfit_batch_cpu, device, subset_size=8)

# 4. Training Loop (150 epochs on the exact same 8 images)
epochs = 1000
loss_history = []

overfit_model.train()
for epoch in range(epochs):
    optimizer.zero_grad(set_to_none=True) # slightly more memory efficient than zero_grad()
    
    # Forward pass with Mixed Precision
    with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=cfg.use_amp):
        images = overfit_batch["image"]
        cliff_cond = overfit_batch["cliff_cond"]
        
        preds = overfit_model(images, cliff_cond)
        
        # Compute loss
        losses = criterion(preds, overfit_batch)
        total_loss = losses['total_loss']
    
    # Backward pass with scaled gradients
    scaler.scale(total_loss).backward()
    
    # Unscale before gradient clipping
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(overfit_model.parameters(), cfg.grad_clip)
    
    # Optimizer step
    scaler.step(optimizer)
    scaler.update()
    
    loss_history.append(total_loss.item())
    
    # Print progress every 10 epochs
    if epoch % 10 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:03d} | Total Loss: {total_loss.item():.4f} | "
              f"2D Loss: {losses['loss_2d_native'].item():.4f} | "
              f"3D Loss: {losses['loss_3d_native'].item():.4f} | "
              f"Loss pose: {losses['loss_pose'].item():.4f} | "
              f"Scale Loss: {losses['loss_scale'].item():.4f} | "
              f"Shape Loss: {losses['loss_shape'].item():.4f} | "
              f"Cam Loss: {losses['loss_cam'].item():.4f}"
              f"Mhr joints Loss: {losses['loss_mhr_joints'].item():.4f}",
              f"Structural Loss: {losses['loss_structural'].item():.4f}")

# 5. Plot the results
plt.figure(figsize=(10, 5))
plt.plot(loss_history, label='Total Loss', color='blue', linewidth=2)
plt.title('1-Batch Overfit Curve (8 Images)', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Total Loss', fontsize=12)
plt.yscale('log') # Log scale helps visualize the exponential decay
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

if loss_history[-1] < loss_history[0] * 0.05:
    print("\n✅ SUCCESS: The model successfully overfit the batch! Gradients are flowing.")
else:
    print("\n❌ WARNING: The loss did not decrease significantly. Check for detached tensors or learning rate issues.")

In [ ]:
# ============================================================
# Cell 10 — HMR Evaluation Metrics (MPJPE & PA-MPJPE)
# ============================================================
import torch
import numpy as np

def batched_procrustes_alignment(pred_pts, gt_pts):
    """
    Computes the optimal Similarity Transform (Scale, Rotation, Translation) 
    to align pred_pts to gt_pts using Batched SVD.
    """
    B, N, D = pred_pts.shape
    
    # 1. Mean center the point clouds
    mu_pred = pred_pts.mean(dim=1, keepdim=True)
    mu_gt = gt_pts.mean(dim=1, keepdim=True)
    
    pred_centered = pred_pts - mu_pred
    gt_centered = gt_pts - mu_gt
    
    # 2. Calculate scale (Frobenius norm)
    norm_pred = torch.linalg.norm(pred_centered, dim=(1,2), keepdim=True)
    norm_gt = torch.linalg.norm(gt_centered, dim=(1,2), keepdim=True)
    
    pred_normalized = pred_centered / torch.clamp(norm_pred, min=1e-8)
    gt_normalized = gt_centered / torch.clamp(norm_gt, min=1e-8)
    
    # 3. Compute covariance matrix H (X^T Y)
    H = torch.bmm(pred_normalized.transpose(1, 2), gt_normalized)
    
    # 4. Singular Value Decomposition (SVD)
    # PyTorch returns U, S, and Vh (where Vh is V^T)
    U, S, Vh = torch.linalg.svd(H)
    
    # 5. Compute Rotation matrix R (U * V^T) - FIXED!
    R = torch.bmm(U, Vh)
    
    # 6. Fix reflection (Strict SO(3) rotation)
    det = torch.linalg.det(R)
    
    # If det < 0, we must flip the sign of the 3rd column of U
    # Shape of det_sign is [B, 1]
    det_sign = torch.where(det < 0, torch.tensor(-1.0, device=pred_pts.device), torch.tensor(1.0, device=pred_pts.device)).unsqueeze(-1)
    
    U_fixed = U.clone()
    U_fixed[:, :, 2] *= det_sign
    
    # Recompute R with the reflection fix
    R_fixed = torch.bmm(U_fixed, Vh)
    
    # 7. Optimal Scale
    # We must apply the same sign fix to the 3rd singular value!
    S_fixed = S.clone()
    S_fixed[:, 2] *= det_sign.squeeze(-1)
    
    scale = S_fixed.sum(dim=-1, keepdim=True).unsqueeze(-1) * (norm_gt / torch.clamp(norm_pred, min=1e-8))
    
    # 8. Apply the transformation
    pred_aligned = scale * torch.bmm(pred_centered, R_fixed) + mu_gt
    
    return pred_aligned

@torch.no_grad()
def evaluate_hmr_batch(preds, targets, cfg, mhr_module):
    """
    Calculates MPJPE and PA-MPJPE for both the 3D Keypoints Head and the MHR Mesh.
    Returns errors in MILLIMETERS (mm).
    """
    metrics = {}
    
    # ---------------------------------------------------------
    # 1. Evaluate the Native 3D Spatial Head (70 Joints)
    # ---------------------------------------------------------
    pred_3d_native = preds["joints_3d"][..., :3]
    gt_3d_native = targets["joints_3d"][..., :3]
    
    # MPJPE requires root-relative alignment (usually joint 0 is the root/pelvis)
    pred_3d_root_rel = pred_3d_native - pred_3d_native[:, 0:1, :]
    gt_3d_root_rel = gt_3d_native - gt_3d_native[:, 0:1, :]
    
    # MPJPE in mm (* 1000)
    err_native = torch.linalg.norm(pred_3d_root_rel - gt_3d_root_rel, dim=-1)
    metrics['Head_3D_MPJPE'] = err_native.mean().item() * 1000.0
    
    # PA-MPJPE in mm
    pred_3d_aligned = batched_procrustes_alignment(pred_3d_native, gt_3d_native)
    err_native_pa = torch.linalg.norm(pred_3d_aligned - gt_3d_native, dim=-1)
    metrics['Head_3D_PA_MPJPE'] = err_native_pa.mean().item() * 1000.0

    # ---------------------------------------------------------
    # 2. Evaluate the MHR Mesh Head (Using the 52 Anchors)
    # ---------------------------------------------------------
    if hasattr(cfg, 'native_mapping_ids') and hasattr(cfg, 'mhr_mapping_ids'):
        # Generate 3D mesh joints in cm
        pred_mhr_raw = mhr_module.get_joints(preds["mhr_params"], preds["shape_params"])[..., :3]
        
        # Convert to Meters and align axes
        pred_mhr_meters = pred_mhr_raw / 100.0
        pred_mhr_meters[..., 1] = -pred_mhr_meters[..., 1]
        pred_mhr_meters[..., 2] = -pred_mhr_meters[..., 2]
        
        # Extract the 52 matching anchors
        pred_mesh_anchors = pred_mhr_meters[:, cfg.mhr_mapping_ids, :]
        gt_mesh_anchors = gt_3d_native[:, cfg.native_mapping_ids, :]
        
        # Mean-Center for MPJPE (To bypass differing origin definitions between rigs)
        pred_mesh_centered = pred_mesh_anchors - pred_mesh_anchors.mean(dim=1, keepdim=True)
        gt_mesh_centered = gt_mesh_anchors - gt_mesh_anchors.mean(dim=1, keepdim=True)
        
        # Mesh MPJPE in mm
        err_mesh = torch.linalg.norm(pred_mesh_centered - gt_mesh_centered, dim=-1)
        metrics['Mesh_MPJPE'] = err_mesh.mean().item() * 1000.0
        
        # Mesh PA-MPJPE in mm (Procrustes naturally handles translation offsets!)
        pred_mesh_aligned = batched_procrustes_alignment(pred_mesh_anchors, gt_mesh_anchors)
        err_mesh_pa = torch.linalg.norm(pred_mesh_aligned - gt_mesh_anchors, dim=-1)
        metrics['Mesh_PA_MPJPE'] = err_mesh_pa.mean().item() * 1000.0
        
    return metrics

# # ==============================================================================
# # SANITY CHECK: Evaluating Your Overfit Model
# # ==============================================================================
# if __name__ == "__main__":
#     print("--- Running Evaluation on Overfit Model ---")
#     overfit_model.eval()
    
#     with torch.no_grad():
#         with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=cfg.use_amp):
#             eval_preds = overfit_model(overfit_batch["image"], overfit_batch["cliff_cond"])
            
#         # Upcast to float32 for rigorous geometric math
#         eval_preds_fp32 = {k: v.float() for k, v in eval_preds.items()}
        
#         metrics = evaluate_hmr_batch(eval_preds_fp32, overfit_batch, cfg, mhr_module)
        
#     print("\n--- Final Metrics on 1-Batch Overfit ---")
#     print(f"3D Keypoint Head MPJPE:      {metrics['Head_3D_MPJPE']:.2f} mm")
#     print(f"3D Keypoint Head PA-MPJPE:   {metrics['Head_3D_PA_MPJPE']:.2f} mm")
#     print("-" * 40)
#     print(f"MHR Mesh Anchors MPJPE:      {metrics['Mesh_MPJPE']:.2f} mm")
#     print(f"MHR Mesh Anchors PA-MPJPE:   {metrics['Mesh_PA_MPJPE']:.2f} mm")
    
#     if metrics['Mesh_PA_MPJPE'] < 10.0:
#         print("\n✅ OUTSTANDING: Mesh PA-MPJPE is under 10mm! Your model has achieved sub-centimeter physical accuracy.")

In [ ]:
# ============================================================
# Cell 11 — The Full Distillation Training Loop (Phase 2/Fresh)
# ============================================================
import torch
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import os
import math
from tqdm.notebook import tqdm

def train_instant_hmr():
    print(f"--- Starting Full Training Pipeline: {cfg.backbone} ---")
    os.makedirs(cfg.log_dir, exist_ok=True)
    
    # --- CONFIGURATION ---
    RESUME_FROM_CHECKPOINT = False  # Set to False to train from scratch
    
    # 1. Initialize Model & Loss
    model = InstantHMRStudent(cfg, pretrained=True).to(device)
    criterion = DistillationLoss(cfg, mhr_module)
    
    # 2. Optimizer & AMP Scaler
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler(device='cuda', enabled=cfg.use_amp)
    
    # --- RESUME / FRESH START LOGIC ---
    best_ckpt_path = os.path.join(cfg.log_dir, "best_student_model.pth")
    start_epoch = 0 
    best_val_metric = float('inf') 
    
    if RESUME_FROM_CHECKPOINT and os.path.exists(best_ckpt_path):
        print(f"🔄 Resuming from checkpoint: {best_ckpt_path}")
        ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=True)
        model.load_state_dict(ckpt["model_state_dict"], strict=False) 
    else:
        print("🚀 Starting training from scratch with fresh weights.")

    # Disable torch.compile() to prevent the LLVM pthread_join crash 
    # and to stop BatchNorm running stats from corrupting during validation.
    # print("⏳ Compiling model with torch.compile()...")
    # model = torch.compile(model, mode="default") 

    # 3. Learning Rate Scheduler
    steps_per_epoch = len(train_loader)
    scheduler = lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=cfg.lr, 
        epochs=cfg.epochs, 
        steps_per_epoch=steps_per_epoch,
        pct_start=0.1
    )
    
    # --- MAIN EPOCH LOOP ---
    for epoch in range(start_epoch, cfg.epochs):
        model.train()
        
        train_metrics = {'total': 0.0, '2d': 0.0, '3d': 0.0, 'pose': 0.0, 'scale': 0.0, 'shape': 0.0, 'mhr': 0.0, 'struct': 0.0}
        valid_train_batches = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs} [Train]")
        for batch in pbar:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=cfg.use_amp):
                preds_fp16 = model(batch["image"], batch["cliff_cond"])
                
            preds_fp32 = {k: v.float() for k, v in preds_fp16.items()}
            losses = criterion(preds_fp32, batch)
            loss = losses['total_loss']
            
            if math.isnan(loss.item()) or math.isinf(loss.item()):
                print("⚠️ WARNING: NaN/Inf loss detected! Skipping step.")
                continue
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            
            # 🛑 Only step the scheduler if the optimizer actually stepped
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            scale_after = scaler.get_scale()
            
            # If the scale didn't drop (meaning no NaNs were found in gradients), step the LR scheduler
            if scale_before <= scale_after:
                scheduler.step()
            
            # Accumulate metrics
            train_metrics['total'] += loss.item()
            train_metrics['2d'] += losses.get('loss_2d_native', torch.tensor(0)).item()
            train_metrics['3d'] += losses.get('loss_3d_native', torch.tensor(0)).item()
            train_metrics['pose'] += losses.get('loss_pose', torch.tensor(0)).item()
            train_metrics['scale'] += losses.get('loss_scale', torch.tensor(0)).item()
            train_metrics['mhr'] += losses.get('loss_mhr_joints', torch.tensor(0)).item()
            train_metrics['struct'] += losses.get('loss_structural', torch.tensor(0)).item()
            valid_train_batches += 1
            
            pbar.set_postfix({
                'Tot': f"{loss.item():.3f}", 
                'MHR': f"{losses.get('loss_mhr_joints', torch.tensor(0)).item():.3f}",
                'Str': f"{losses.get('loss_structural', torch.tensor(0)).item():.3f}"
            })
            
        pbar.close() 
        avg_train = {k: v / max(valid_train_batches, 1) for k, v in train_metrics.items()}
        
        # --- VALIDATION LOOP ---
        model.eval()
        val_metrics = {'total': 0.0, '2d': 0.0, '3d': 0.0, 'mhr': 0.0, 'struct': 0.0}
        
        hmr_trackers = {'Head_3D_MPJPE': 0.0, 'Head_3D_PA_MPJPE': 0.0, 'Mesh_MPJPE': 0.0, 'Mesh_PA_MPJPE': 0.0}
        valid_val_batches = 0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{cfg.epochs} [Val]", leave=False):
                batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
                
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=cfg.use_amp):
                    preds_fp16 = model(batch["image"], batch["cliff_cond"])
                
                preds_fp32 = {k: v.float() for k, v in preds_fp16.items()}
                losses = criterion(preds_fp32, batch)
                loss = losses['total_loss']
                
                if math.isnan(loss.item()) or math.isinf(loss.item()):
                    continue
                    
                val_metrics['total'] += loss.item()
                val_metrics['2d'] += losses.get('loss_2d_native', torch.tensor(0)).item()
                val_metrics['3d'] += losses.get('loss_3d_native', torch.tensor(0)).item()
                val_metrics['mhr'] += losses.get('loss_mhr_joints', torch.tensor(0)).item()
                val_metrics['struct'] += losses.get('loss_structural', torch.tensor(0)).item()
                
                batch_hmr_metrics = evaluate_hmr_batch(preds_fp32, batch, cfg, mhr_module)
                for k in hmr_trackers.keys():
                    hmr_trackers[k] += batch_hmr_metrics.get(k, 0.0)
                    
                valid_val_batches += 1
                    
        avg_val = {k: v / max(valid_val_batches, 1) for k, v in val_metrics.items()}
        avg_hmr = {k: v / max(valid_val_batches, 1) for k, v in hmr_trackers.items()}
        
        # --- DETAILED EPOCH SUMMARY ---
        print(f"\n📈 Epoch {epoch+1} Summary | LR: {scheduler.get_last_lr()[0]:.2e}")
        print(f"   [Train Loss] Tot: {avg_train['total']:.4f} | MHR: {avg_train['mhr']:.4f} | Struct: {avg_train['struct']:.4f} | 2D: {avg_train['2d']:.4f} | 3D: {avg_train['3d']:.4f}")
        print(f"   [Val Loss]   Tot: {avg_val['total']:.4f} | MHR: {avg_val['mhr']:.4f} | Struct: {avg_val['struct']:.4f} | 2D: {avg_val['2d']:.4f} | 3D: {avg_val['3d']:.4f}")
        print(f"   📏 [Val Metrics]  Mesh PA-MPJPE: {avg_hmr['Mesh_PA_MPJPE']:.1f} mm | Mesh MPJPE: {avg_hmr['Mesh_MPJPE']:.1f} mm")
        print(f"   📏 [Val Metrics]  Head PA-MPJPE: {avg_hmr['Head_3D_PA_MPJPE']:.1f} mm | Head MPJPE: {avg_hmr['Head_3D_MPJPE']:.1f} mm")
        
        # --- CHECKPOINTING (Now based on Mesh PA-MPJPE!) ---
        current_metric = avg_hmr['Mesh_PA_MPJPE']
        if current_metric < best_val_metric and valid_val_batches > 0:
            best_val_metric = current_metric
            save_path = os.path.join(cfg.log_dir, "best_student_model_v3.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_pa_mpjpe': best_val_metric,
            }, save_path)
            print(f"💾 New Best Model saved! Mesh PA-MPJPE dropped to {best_val_metric:.1f} mm.")

    print("\n🎉 Training Complete!")

# Kick off the training!
if __name__ == "__main__":
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    
    train_instant_hmr()

In [ ]:
# ============================================================
# Cell 13 — Validation Visualization (The Reveal!)
# ============================================================
import torch
import matplotlib.pyplot as plt
import os

print("--- Running Inference Visualization ---")

# 1. Load the Best Saved Model
best_model = InstantHMRStudent(cfg, pretrained=False).to(device)
checkpoint_path = os.path.join(cfg.log_dir, "best_student_model_v3.pth")

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    
    # --- THE FIX: Clean the torch.compile() prefix ---
    clean_state_dict = {}
    for k, v in checkpoint['model_state_dict'].items():
        # Remove '_orig_mod.' prefix if it exists
        new_key = k.replace('_orig_mod.', '')
        clean_state_dict[new_key] = v
        
    # Load the cleaned dictionary
    best_model.load_state_dict(clean_state_dict, strict=False)
    print(f"✅ Loaded Best Model from Epoch {checkpoint.get('epoch', 0)+1} with PA-MPJPE: {checkpoint.get('val_pa_mpjpe', 0):.1f} mm")
else:
    print("⚠️ No checkpoint found! Using current model in memory.")
    best_model = model

best_model.eval()

# 2. Grab a Validation Batch
val_batch = next(iter(val_loader))
images = val_batch["image"].to(device)
cliff_cond = val_batch["cliff_cond"].to(device)
gt_2d = val_batch["joints_2d"].cpu()

# 3. Run Inference
with torch.no_grad():
    with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=cfg.use_amp):
        preds = best_model(images, cliff_cond)

# Extract only the X,Y coordinates
pred_2d = preds["joints_2d"][..., :2].cpu()

# 4. Math Helper: Un-normalize coordinates from [-1, 1] back to [0, 224] pixels
def unnorm_coords(coords, size=224.0):
    return ((coords + 1.0) / 2.0) * size

gt_2d_px = unnorm_coords(gt_2d[..., :2], cfg.image_size)
pred_2d_px = unnorm_coords(pred_2d, cfg.image_size)

# 5. Image Helper: Un-normalize ImageNet pixels for matplotlib
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# 6. Plot a 2x2 Grid of Predictions
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

for i in range(4): # Plot the first 4 images in the batch
    # Prepare image
    img_unnorm = images[i].cpu() * std + mean
    img_np = img_unnorm.permute(1, 2, 0).numpy().clip(0, 1)
    
    # Plot Image
    axes[i].imshow(img_np)
    
    # Plot Ground Truth (Teacher) - Lime Green Dots
    axes[i].scatter(gt_2d_px[i, :, 0], gt_2d_px[i, :, 1], 
                    c='lime', s=25, edgecolors='black', label='Teacher (GT)' if i==0 else "")
    
    # Plot Predictions (Student) - Red Crosses
    axes[i].scatter(pred_2d_px[i, :, 0], pred_2d_px[i, :, 1], 
                    c='red', s=35, marker='X', edgecolors='black', linewidths=0.5, label='Student' if i==0 else "")
    
    axes[i].axis('off')
    axes[i].set_title(f"Validation Sample {i+1}", fontsize=12)

fig.legend(loc='upper center', ncol=2, fontsize=14, bbox_to_anchor=(0.5, 1.05))
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 14 — Inference Speed Benchmark (Fixed Loader)
# ============================================================
import time
import os
import torch

print("--- Running Inference Speed Benchmark ---")

@torch.no_grad()
def benchmark_fps(model, image_size=224, batch_size=1, n_warmup=50, n_iters=200):
    """Benchmark model throughput on GPU."""
    model.eval()
    # Dummy inputs matching our architecture
    dummy_img = torch.randn(batch_size, 3, image_size, image_size, device=device)
    dummy_cliff = torch.randn(batch_size, 3, device=device) # [cx_norm, cy_norm, b_scale]

    # Warmup (Wraps the forward pass in AMP for real-world float16 speed)
    with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=cfg.use_amp):
        for _ in range(n_warmup):
            _ = model(dummy_img, dummy_cliff)
            
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # Timed run
    t0 = time.perf_counter()
    with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=cfg.use_amp):
        for _ in range(n_iters):
            _ = model(dummy_img, dummy_cliff)
            
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
    elapsed = time.perf_counter() - t0

    fps = n_iters * batch_size / elapsed
    ms_per_image = elapsed / (n_iters * batch_size) * 1000
    return fps, ms_per_image

# 1. Instantiate the Model
benchmark_model = InstantHMRStudent(cfg, pretrained=False).to(device)

# 2. Load best checkpoint with the compile fix!
best_ckpt_path = os.path.join(cfg.log_dir, "best_student_model_v3.pth")
if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=True)
    
    # --- Clean the torch.compile() prefix ---
    clean_state_dict = {}
    for k, v in ckpt['model_state_dict'].items():
        new_key = k.replace('_orig_mod.', '')
        clean_state_dict[new_key] = v
        
    benchmark_model.load_state_dict(clean_state_dict, strict=False)
    print(f"✅ Loaded best model from epoch {ckpt.get('epoch', 0)+1} (PA-MPJPE={ckpt.get('val_pa_mpjpe', 0):.1f} mm)")
else:
    print("⚠️ No checkpoint found! Benchmarking with random weights.")

# 3. Benchmark at different batch sizes
fps_b1, ms_b1 = benchmark_fps(benchmark_model, batch_size=1)
fps_b16, ms_b16 = benchmark_fps(benchmark_model, batch_size=16)
fps_b64, ms_b64 = benchmark_fps(benchmark_model, batch_size=64)

# 4. Compute total parameters
total_params = sum(p.numel() for p in benchmark_model.parameters())

print(f"\n🚀 Inference Speed (RTX {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}):")
print(f"  Batch=1:  {fps_b1:.0f} FPS  ({ms_b1:.1f} ms/image)")
print(f"  Batch=16: {fps_b16:.0f} FPS  ({ms_b16:.1f} ms/image)")
print(f"  Batch=64: {fps_b64:.0f} FPS  ({ms_b64:.1f} ms/image)")
print(f"\n📊 Comparison:")
print(f"  SAM3D Teacher: ~3 FPS (670M params)")
print(f"  Student:       {fps_b1:.0f} FPS ({total_params/1e6:.1f}M params) → {fps_b1/3:.0f}x faster")

In [ ]:
# ============================================================
# Cell 15 — Export & Static Quantization (FP32, FP16, INT8)
# ============================================================
import os
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

print("--- Exporting and Quantizing Full Model for Deployment ---")

# --- 1. Deploy Wrapper (full outputs) ---
class HMRDeployWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, image, cliff_cond):
        out = self.model(image, cliff_cond)
        return (out["mhr_params"], out["shape_params"], out["cam_trans"], 
                out["joints_2d"], out["joints_3d"])

# --- 2. INT8 Calibration Data Reader ---
try:
    from onnxruntime.quantization import (
        CalibrationDataReader, quantize_static,
        QuantType, QuantFormat, CalibrationMethod,
    )
    
    class HMRCalibrationDataReader(CalibrationDataReader):
        def __init__(self, dataloader, max_samples=100):
            self.data_list = []
            
            print(f"Extracting {max_samples} individual images for INT8 calibration...")
            for batch in dataloader:
                images = batch["image"].numpy().astype(np.float32)
                cliffs = batch["cliff_cond"].numpy().astype(np.float32)
                
                for b_idx in range(images.shape[0]):
                    if len(self.data_list) >= max_samples:
                        break
                    self.data_list.append({
                        "image": images[b_idx:b_idx+1],
                        "cliff_cond": cliffs[b_idx:b_idx+1]
                    })
                if len(self.data_list) >= max_samples:
                    break
                    
            self.enum_data = iter(self.data_list)

        def get_next(self):
            return next(self.enum_data, None)

except ImportError:
    print("⚠️ 'onnxruntime' is not installed. INT8 quantization will fail.")


def topological_sort_onnx(model):
    """Sort ONNX graph nodes in topological order.
    
    The ORT float16 converter with keep_io_types=True inserts Cast nodes at graph
    inputs but doesn't re-sort the graph, breaking topological order. This fixes it.
    """
    available = set()
    for inp in model.graph.input:
        available.add(inp.name)
    for init in model.graph.initializer:
        available.add(init.name)
    
    remaining = list(model.graph.node)
    sorted_nodes = []
    
    while remaining:
        progress = False
        for node in remaining:
            if all(inp in available or inp == '' for inp in node.input):
                sorted_nodes.append(node)
                remaining.remove(node)
                for out in node.output:
                    available.add(out)
                progress = True
                break
        if not progress:
            sorted_nodes.extend(remaining)
            break
    
    del model.graph.node[:]
    model.graph.node.extend(sorted_nodes)
    return model


# --- 3. Export & Quantize ---
export_dir = Path(cfg.log_dir) / "export"
export_dir.mkdir(parents=True, exist_ok=True)

onnx_fp32_path = export_dir / "sam3d_full_fp32.onnx"
onnx_fp16_path = export_dir / "sam3d_full_fp16.onnx"
onnx_int8_path = export_dir / "sam3d_full_int8.onnx"

output_keys = ["mhr_params", "shape_params", "cam_trans", "joints_2d", "joints_3d"]

export_target_model = InstantHMRStudent(cfg, pretrained=False).to(device)

best_ckpt_path = os.path.join(cfg.log_dir, "best_student_model_v3.pth")

if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=True)
    clean_state_dict = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
    export_target_model.load_state_dict(clean_state_dict, strict=False)
    print(f"✅ Loaded best model from epoch {ckpt.get('epoch', 0)+1}.\n")
    
    deploy_model = HMRDeployWrapper(export_target_model)
    deploy_model.eval()
    
    dummy_img = torch.randn(1, 3, cfg.image_size, cfg.image_size, device=device)
    dummy_cliff = torch.randn(1, 3, device=device)
    
    dynamic_axes = {"image": {0: "batch"}, "cliff_cond": {0: "batch"}}
    for k in output_keys:
        dynamic_axes[k] = {0: "batch"}

    # A. Export FP32
    try:
        import onnx
        torch.onnx.export(
            deploy_model, (dummy_img, dummy_cliff), str(onnx_fp32_path),
            input_names=["image", "cliff_cond"], output_names=output_keys,
            dynamic_axes=dynamic_axes, opset_version=17
        )
        print(f"✅ FP32 saved: {onnx_fp32_path.name} | Size: {onnx_fp32_path.stat().st_size / 1e6:.1f} MB")
    except Exception as e:
        print(f"❌ FP32 export failed: {e}")

    # B. Export FP16 — uses ORT's converter which correctly updates Cast node `to` attributes
    #    + topological sort to fix node ordering after keep_io_types Cast insertion
    try:
        from onnxruntime.transformers.float16 import convert_float_to_float16
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model_fp32 = onnx.load(str(onnx_fp32_path))
            model_fp16 = convert_float_to_float16(model_fp32, keep_io_types=True)
            model_fp16 = topological_sort_onnx(model_fp16)
            onnx.save(model_fp16, str(onnx_fp16_path))
        onnx.checker.check_model(onnx.load(str(onnx_fp16_path)), full_check=True)
        print(f"✅ FP16 saved: {onnx_fp16_path.name} | Size: {onnx_fp16_path.stat().st_size / 1e6:.1f} MB")
    except ImportError:
        print("⚠️ FP16 skipped (onnxruntime.transformers not available)")
    except Exception as e:
        print(f"❌ FP16 conversion failed: {e}")

    # C. Export INT8 — selective: only quantize Conv ops (backbone), skip transformer/heads
    #    Naive quantization of all ops destroys accuracy (Softmax, LayerNorm, attention MatMuls
    #    need FP32 precision) and is slower due to unfused QDQ overhead blocking NCHWc optimization.
    try:
        calib_reader = HMRCalibrationDataReader(train_loader, max_samples=100)
        calib_reader.enum_data = iter(calib_reader.data_list)
        
        # Collect transformer + head + projection node names to exclude from quantization
        # (defense-in-depth: op_types_to_quantize=['Conv'] already excludes them since they
        #  have zero Conv ops, but explicit exclusion prevents issues if the model changes)
        fp32_model = onnx.load(str(onnx_fp32_path))
        nodes_to_exclude = []
        for node in fp32_model.graph.node:
            name = node.name or ''
            if any(kw in name for kw in ['/model/transformer/', '/model/head_',
                                          '/model/feat_proj', '/model/cond_proj']):
                nodes_to_exclude.append(name)
        del fp32_model
        print(f"  Excluding {len(nodes_to_exclude)} transformer/head nodes from quantization")
        
        quantize_static(
            model_input=str(onnx_fp32_path),
            model_output=str(onnx_int8_path),
            calibration_data_reader=calib_reader,
            quant_format=QuantFormat.QDQ,
            op_types_to_quantize=['Conv'],       # Only quantize Conv (backbone)
            per_channel=True,                     # Better accuracy for Conv weights
            reduce_range=False,                   # Not needed with VNNI
            activation_type=QuantType.QInt8,
            weight_type=QuantType.QInt8,
            nodes_to_exclude=nodes_to_exclude,
            calibrate_method=CalibrationMethod.Percentile,
            extra_options={
                'CalibTensorRangeSymmetric': True,
            },
        )
        print(f"✅ INT8 saved: {onnx_int8_path.name} | Size: {onnx_int8_path.stat().st_size / 1e6:.1f} MB")
    except NameError:
        print("⚠️ INT8 skipped (train_loader not available)")
    except Exception as e:
        import traceback
        print(f"❌ INT8 conversion failed: {e}")
        traceback.print_exc()

    # D. Save PyTorch state dict
    sd_path = export_dir / "sam3d_student_state_dict.pth"
    torch.save(export_target_model.state_dict(), sd_path)
    print(f"\n✅ PyTorch state dict saved: {sd_path.name}")
    
else:
    print("⚠️ No checkpoint found at:", best_ckpt_path)

In [ ]:
# ============================================================
# Cell 17 — ONNX Profiler (FP32, FP16, INT8) x (CPU, GPU)
# ============================================================
import time
import numpy as np
from pathlib import Path

print("--- ONNX Hardware Profiler ---")

try:
    import onnxruntime as ort
    ONNX_AVAILABLE = True
except ImportError:
    ONNX_AVAILABLE = False
    print("⚠️ Missing: pip install onnxruntime-gpu")

if ONNX_AVAILABLE:
    def profile_onnx_model(onnx_path, provider, batch_size=1, warmup_iters=20, benchmark_iters=100):
        """Profile latency for a specific ONNX model and execution provider."""
        
        sess_options = ort.SessionOptions()
        sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        sess_options.log_severity_level = 3
        
        try:
            session = ort.InferenceSession(str(onnx_path), sess_options=sess_options, providers=[provider])
        except Exception as e:
            return None, None, f"Error: {e}"
            
        active_provider = session.get_providers()[0]
        if active_provider != provider:
            return None, None, f"Fallback! Requested {provider} but got {active_provider}"

        ort_inputs = {}
        dtype_map = {'tensor(float)': np.float32, 'tensor(float16)': np.float16}
        
        for inp in session.get_inputs():
            shape = [batch_size if isinstance(dim, str) else dim for dim in inp.shape]
            np_dtype = dtype_map.get(inp.type, np.float32)
            ort_inputs[inp.name] = np.random.randn(*shape).astype(np_dtype)

        for _ in range(warmup_iters):
            session.run(None, ort_inputs)

        t0 = time.perf_counter()
        for _ in range(benchmark_iters):
            session.run(None, ort_inputs)
        t1 = time.perf_counter()

        total_time = t1 - t0
        avg_ms = (total_time / benchmark_iters) * 1000.0
        fps = (benchmark_iters * batch_size) / total_time
        return avg_ms, fps, "OK"

    # --- Setup ---
    export_dir = Path(cfg.log_dir) / "export"
    precisions = ["fp32", "fp16", "int8"]

    available_providers = ort.get_available_providers()
    providers_to_test = ['CPUExecutionProvider']
    if 'CUDAExecutionProvider' in available_providers:
        providers_to_test.append('CUDAExecutionProvider')
    else:
        print("⚠️ CUDAExecutionProvider not found. Profiling CPU only.")
        print("   Fix: pip uninstall onnxruntime onnxruntime-gpu -y && pip install onnxruntime-gpu")

    print(f"\n{'Precision':<6} | {'Size':<7} | {'HW':<4} | {'Latency':<12} | {'FPS':<8} | Notes")
    print("-" * 75)

    for prec in precisions:
        model_path = export_dir / f"sam3d_full_{prec}.onnx"
        
        if not model_path.exists():
            print(f"{prec:<6} | ---     | ---  | {'Not Found':<12} | ---")
            continue
            
        size_mb = f"{model_path.stat().st_size / 1e6:.1f}MB"
            
        for provider in providers_to_test:
            hw_tag = "GPU" if "CUDA" in provider else "CPU"
            
            # FP16 on CPU: not supported (no native x86 FP16 compute)
            if prec == "fp16" and provider == "CPUExecutionProvider":
                print(f"{prec:<6} | {size_mb:<7} | {hw_tag:<4} | {'N/A':<12} | ---      | x86 has no FP16 ALU")
                continue
            
            note = ""
            if prec == "fp16" and "CUDA" in provider:
                note = "Best for GPU deploy"
            elif prec == "int8" and "CPU" in provider:
                note = "Conv-only quant, for edge/NPU"
            elif prec == "int8" and "CUDA" in provider:
                note = "Needs TensorRT EP for gains"
            
            ms, fps, status = profile_onnx_model(model_path, provider)
            
            if ms is not None:
                print(f"{prec:<6} | {size_mb:<7} | {hw_tag:<4} | {ms:>7.2f} ms   | {fps:>6.1f}  | {note}")
            else:
                print(f"{prec:<6} | {size_mb:<7} | {hw_tag:<4} | {'Failed':<12} | ---      | {status}")

    print("-" * 75)
    print("\nDeployment guide:")
    print("  GPU (CUDA EP)     → FP16 (best latency + half memory)")
    print("  GPU (TensorRT EP) → INT8 (if TRT available, even faster)")
    print("  CPU (x86 VNNI)    → FP32 or INT8 (Conv-only quant, test both)")
    print("  Mobile NPU        → INT8 with NNAPI/QNN EP (native INT8 HW)")

In [ ]:
# ============================================================
# Cell 18 — ONNX Accuracy Comparison (FP32 vs FP16 vs INT8)
# ============================================================
import numpy as np
import torch
from pathlib import Path
from tqdm.notebook import tqdm

print("--- ONNX Quantization Accuracy Comparison ---")

try:
    import onnxruntime as ort
    ONNX_AVAILABLE = True
except ImportError:
    ONNX_AVAILABLE = False
    print("⚠️ Missing: pip install onnxruntime-gpu")

if ONNX_AVAILABLE:
    export_dir = Path(cfg.log_dir) / "export"
    output_names = ["mhr_params", "shape_params", "cam_trans", "joints_2d", "joints_3d"]
    
    models_to_eval = {
        "FP32": export_dir / "sam3d_full_fp32.onnx",
        "FP16": export_dir / "sam3d_full_fp16.onnx",
        "INT8": export_dir / "sam3d_full_int8.onnx",
    }
    
    # Filter to existing models only
    models_to_eval = {k: v for k, v in models_to_eval.items() if v.exists()}
    print(f"Found models: {', '.join(models_to_eval.keys())}")
    
    # Use GPU if available, otherwise CPU (FP16 needs GPU)
    available_providers = ort.get_available_providers()
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in available_providers else ['CPUExecutionProvider']
    
    results = {}
    
    for model_name, model_path in models_to_eval.items():
        print(f"\nEvaluating {model_name} ({model_path.stat().st_size / 1e6:.1f} MB)...")
        
        sess_options = ort.SessionOptions()
        sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        sess_options.log_severity_level = 3
        
        try:
            session = ort.InferenceSession(str(model_path), sess_options=sess_options, providers=providers)
        except Exception as e:
            print(f"  ❌ Failed to load: {e}")
            continue
        
        active_provider = session.get_providers()[0]
        print(f"  Running on: {active_provider}")
        
        # Accumulate metrics across all val batches
        hmr_trackers = {'Head_3D_MPJPE': 0.0, 'Head_3D_PA_MPJPE': 0.0, 'Mesh_MPJPE': 0.0, 'Mesh_PA_MPJPE': 0.0}
        valid_batches = 0
        
        for batch in tqdm(val_loader, desc=f"  {model_name}", leave=False):
            images_np = batch["image"].numpy().astype(np.float32)
            cliff_np = batch["cliff_cond"].numpy().astype(np.float32)
            
            ort_inputs = {"image": images_np, "cliff_cond": cliff_np}
            ort_outputs = session.run(None, ort_inputs)
            
            # Convert ONNX outputs to PyTorch tensor dict
            preds = {}
            for name, arr in zip(output_names, ort_outputs):
                preds[name] = torch.from_numpy(arr.astype(np.float32)).to(device)
            
            # Move GT targets to device
            targets = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            batch_metrics = evaluate_hmr_batch(preds, targets, cfg, mhr_module)
            for k in hmr_trackers:
                hmr_trackers[k] += batch_metrics.get(k, 0.0)
            valid_batches += 1
        
        avg_metrics = {k: v / max(valid_batches, 1) for k, v in hmr_trackers.items()}
        results[model_name] = avg_metrics
        print(f"  ✅ {valid_batches} batches evaluated")
    
    # --- Print Comparison Table ---
    print(f"\n{'='*72}")
    print(f"{'Metric':<22} |", end="")
    for name in results:
        print(f" {name:>10}", end=" |")
    
    # Show degradation vs FP32
    if "FP32" in results and len(results) > 1:
        print(f" {'vs FP32':>10}", end="")
    print()
    print("-" * 72)
    
    metric_labels = {
        'Head_3D_MPJPE': 'Head 3D MPJPE (mm)',
        'Head_3D_PA_MPJPE': 'Head 3D PA-MPJPE (mm)',
        'Mesh_MPJPE': 'Mesh MPJPE (mm)',
        'Mesh_PA_MPJPE': 'Mesh PA-MPJPE (mm)',
    }
    
    for key, label in metric_labels.items():
        print(f"{label:<22} |", end="")
        for name in results:
            print(f" {results[name][key]:>10.2f}", end=" |")
        
        # Show delta vs FP32 for non-FP32 models
        if "FP32" in results and len(results) > 1:
            deltas = []
            for name in results:
                if name != "FP32":
                    delta = results[name][key] - results["FP32"][key]
                    deltas.append(f"{delta:+.2f}")
            print(f" {', '.join(deltas):>10}", end="")
        print()
    
    print(f"{'='*72}")